# PY-08 | Coordinate Reference Systems (CRS) με GeoPandas

Στο PY-07 δημιουργήσαμε ένα GeoPackage που συνδυάζει τα όρια των **333 Δήμων** με τα στατιστικά ανεργίας της ΕΛΣΤΑΤ. Σε αυτό το μάθημα χρησιμοποιούμε το ίδιο dataset για να καταλάβουμε τι σημαίνει πραγματικά ένα **Coordinate Reference System (CRS)**.

> **Κεντρική ιδέα:** μια γεωμετρία αποκτά γεωγραφικό νόημα μόνο όταν γνωρίζουμε σε ποιο σύστημα συντεταγμένων ανήκουν οι αριθμοί της.


## 1. Στόχοι του μαθήματος

Στο τέλος του PY-08 θα μπορούμε να:

- εξηγούμε τι περιγράφει ένα CRS,
- ξεχωρίζουμε geographic και projected CRS,
- επιθεωρούμε EPSG code και μονάδες με GeoPandas,
- καταλαβαίνουμε γιατί οι ίδιες θέσεις μπορούν να έχουν διαφορετικές αριθμητικές συντεταγμένες,
- μετασχηματίζουμε σωστά ένα GeoDataFrame με `.to_crs()`,
- εξηγούμε τη διαφορά `.set_crs()` και `.to_crs()`,
- αναγνωρίζουμε ένα dataset στο οποίο έχει δηλωθεί λάθος CRS,
- υπολογίζουμε έκταση σε κατάλληλο projected CRS.


## 2. Imports και path

Χρησιμοποιούμε το GeoPackage που δημιουργήθηκε στο τέλος του PY-07.

```text
Data/
└── processed/
    └── elstat_unemployment_2021_municipalities.gpkg
```


In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt

INPUT_PATH = Path(
    "Data/processed/elstat_unemployment_2021_municipalities.gpkg"
)
LAYER_NAME = "municipal_unemployment_2021"


## 3. Φόρτωση του GeoPackage από το PY-07

Το GeoPackage περιέχει ήδη attributes, geometry και CRS, επομένως δεν χρειάζεται να επαναλάβουμε το attribute join.


In [ ]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Δεν βρέθηκε το GeoPackage: {INPUT_PATH}"
    )

municipalities = gpd.read_file(
    INPUT_PATH,
    layer=LAYER_NAME,
)

print("Type:", type(municipalities))
print("Shape:", municipalities.shape)
municipalities.head()


## 4. Πρώτος έλεγχος του CRS

Το `.crs` δεν είναι απλώς ένα label. Είναι ένα αντικείμενο με πληροφορίες για το σύστημα αναφοράς, τους άξονες και τις μονάδες των συντεταγμένων.


In [ ]:
crs = municipalities.crs

print(crs)
print("EPSG:", crs.to_epsg())
print("Projected:", crs.is_projected)
print("Geographic:", crs.is_geographic)

assert crs.to_epsg() == 2100


## 5. Τι σημαίνουν οι μονάδες του CRS;

Το **EPSG:2100 — GGRS87 / Greek Grid** είναι projected CRS. Οι άξονές του είναι Easting και Northing και η μονάδα είναι το **metre**.


In [ ]:
for axis in crs.axis_info:
    print(
        "Axis:", axis.name,
        "| Direction:", axis.direction,
        "| Unit:", axis.unit_name,
    )


## 6. Οι συντεταγμένες είναι απλώς αριθμοί χωρίς το CRS

Ας πάρουμε ένα σημείο μέσα στον πρώτο Δήμο. Στο EPSG:2100 οι τιμές του είναι σε μέτρα. Δεν είναι longitude και latitude.


In [ ]:
sample_name = municipalities.loc[0, "NAME_GR"]

sample_point_2100 = gpd.GeoSeries(
    [municipalities.geometry.iloc[0].representative_point()],
    crs=municipalities.crs,
)

point_2100 = sample_point_2100.iloc[0]

print("Municipality:", sample_name)
print("Easting:", round(point_2100.x, 2))
print("Northing:", round(point_2100.y, 2))


## 7. Geographic και projected CRS

Ένα **geographic CRS** περιγράφει θέσεις με γωνιακές συντεταγμένες, συνήθως longitude/latitude σε μοίρες. Ένα **projected CRS** μεταφέρει την καμπύλη επιφάνεια της Γης σε επίπεδο σύστημα x/y, ώστε οι συντεταγμένες να μπορούν να εκφράζονται σε γραμμικές μονάδες όπως μέτρα.

Για το μάθημα θα συγκρίνουμε:

- **EPSG:2100** — projected CRS για την Ελλάδα, μονάδα metre,
- **EPSG:4326** — geographic CRS, longitude/latitude σε degrees.


## 8. Reprojection με `.to_crs()`

Η `.to_crs()` **μετασχηματίζει τις ίδιες γεωγραφικές θέσεις** σε νέο σύστημα συντεταγμένων. Οι αριθμητικές τιμές των coordinates αλλάζουν, αλλά ο Δήμος παραμένει στο ίδιο πραγματικό μέρος.


In [ ]:
municipalities_4326 = municipalities.to_crs(epsg=4326)

print("Original CRS:", municipalities.crs)
print("New CRS:", municipalities_4326.crs)
print("New EPSG:", municipalities_4326.crs.to_epsg())
print("Geographic:", municipalities_4326.crs.is_geographic)


## 9. Το ίδιο σημείο με διαφορετικές συντεταγμένες

Μετασχηματίζουμε **το ίδιο sample point** από EPSG:2100 σε EPSG:4326. Αυτό κάνει τη διαφορά πολύ καθαρή: η θέση δεν αλλάζει, αλλά η αριθμητική αναπαράσταση αλλάζει.


In [ ]:
sample_point_4326 = sample_point_2100.to_crs(epsg=4326)
point_4326 = sample_point_4326.iloc[0]

print("EPSG:2100:")
print("  x =", round(point_2100.x, 2))
print("  y =", round(point_2100.y, 2))

print("EPSG:4326:")
print("  longitude =", round(point_4326.x, 6))
print("  latitude  =", round(point_4326.y, 6))


## 10. Σύγκριση των coordinate ranges

Το `.total_bounds` επιστρέφει:

```text
[min_x, min_y, max_x, max_y]
```

Στο EPSG:2100 βλέπουμε μεγάλες τιμές σε μέτρα. Στο EPSG:4326 βλέπουμε longitude/latitude σε μοίρες.


In [ ]:
print("EPSG:2100 bounds:")
print(municipalities.total_bounds)

print()
print("EPSG:4326 bounds:")
print(municipalities_4326.total_bounds)


## 11. Ο χάρτης μπορεί να μοιάζει ίδιος — οι άξονες όμως όχι

Αν αφήσουμε τους άξονες ορατούς, βλέπουμε ότι η Ελλάδα σχεδιάζεται με εντελώς διαφορετικές αριθμητικές συντεταγμένες στα δύο CRS.


In [ ]:
ax = municipalities.plot(
    figsize=(8, 8),
    facecolor="#d9eaf7",
    edgecolor="#4d4d4d",
    linewidth=0.25,
)
ax.set_title("Greece in EPSG:2100 — metres")
ax.set_xlabel("Easting")
ax.set_ylabel("Northing")
plt.show()


In [ ]:
ax = municipalities_4326.plot(
    figsize=(8, 8),
    facecolor="#d9eaf7",
    edgecolor="#4d4d4d",
    linewidth=0.25,
)
ax.set_title("Greece in EPSG:4326 — degrees")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.show()


## 12. `.set_crs()` και `.to_crs()` δεν κάνουν το ίδιο πράγμα

Η διαφορά είναι θεμελιώδης:

- `.set_crs(...)` **δηλώνει** ποιο CRS πρέπει να θεωρήσουμε ότι έχουν οι υπάρχοντες αριθμοί. Δεν μετασχηματίζει τις coordinates.
- `.to_crs(...)` **μετασχηματίζει** τις coordinates από το γνωστό CRS σε ένα άλλο CRS.

```text
set_crs() → αλλάζει/ορίζει metadata
 to_crs() → μετασχηματίζει coordinates
```

Χρησιμοποιούμε `set_crs()` όταν γνωρίζουμε το πραγματικό CRS ενός dataset αλλά το CRS metadata λείπει ή είναι λάθος. Δεν το χρησιμοποιούμε για reprojection.


## 13. Σκόπιμα λάθος CRS

Για να δούμε τον κίνδυνο, δημιουργούμε αντίγραφο και **δηλώνουμε ψευδώς** ότι οι meter coordinates του EPSG:2100 είναι longitude/latitude του EPSG:4326.

Το `allow_override=True` παρακάμπτει την προστασία του GeoPandas. Το χρησιμοποιούμε εδώ μόνο για εκπαιδευτική επίδειξη.


In [ ]:
wrong_crs = municipalities.copy()
wrong_crs = wrong_crs.set_crs(
    epsg=4326,
    allow_override=True,
)

print("Declared CRS:", wrong_crs.crs)
print("Wrong bounds:", wrong_crs.total_bounds)
print("Correct EPSG:4326 bounds:", municipalities_4326.total_bounds)


## 14. Τι ακριβώς πήγε στραβά;

Το `set_crs()` δεν άλλαξε τα coordinates. Απλώς είπε στο GeoPandas να ερμηνεύσει τους ίδιους αριθμούς ως μοίρες. Τιμές εκατοντάδων χιλιάδων ή εκατομμυρίων δεν μπορούν να είναι κανονικά longitude/latitude.

Αυτό είναι επικίνδυνο επειδή ένα polygon μπορεί ακόμη να **φαίνεται σαν σωστό σχήμα** όταν το σχεδιάσουμε μόνο του. Η αριθμητική γεωγραφική του θέση, όμως, είναι λανθασμένη.


In [ ]:
original_geometry = municipalities.geometry.iloc[0]
wrong_geometry = wrong_crs.geometry.iloc[0]
correct_4326_geometry = municipalities_4326.geometry.iloc[0]

print(
    "Same coordinates after set_crs():",
    original_geometry == wrong_geometry
)
print(
    "Same coordinates after to_crs():",
    original_geometry == correct_4326_geometry
)


## 15. CRS και υπολογισμός έκτασης

Για planar area θέλουμε κατάλληλο projected CRS με γραμμικές μονάδες. Εδώ το EPSG:2100 έχει μονάδα metre, άρα η `.area` επιστρέφει τετραγωνικά μέτρα.

```text
1 km² = 1,000,000 m²
```


In [ ]:
municipalities_with_area = municipalities.copy()

municipalities_with_area["area_km2"] = (
    municipalities_with_area.geometry.area / 1_000_000
)

municipalities_with_area[
    ["NAME_GR", "area_km2"]
].head()


## 16. Οι μεγαλύτεροι Δήμοι ως έλεγχος

Ένας πραγματικός υπολογισμός είναι και ευκαιρία για sanity check των αποτελεσμάτων.


In [ ]:
largest_municipalities = municipalities_with_area.nlargest(
    5,
    "area_km2",
)

largest_municipalities[
    ["NAME_GR", "area_km2"]
]


## 17. Γιατί όχι area σε EPSG:4326;

Το EPSG:4326 χρησιμοποιεί **degrees**, όχι metres. Η planar `.area` του GeoPandas/Shapely δεν μετατρέπει αυτόματα τις μοίρες σε km².

Άρα αυτό **δεν** είναι σωστό workflow για έκταση σε km²:

```python
# ΜΗΝ το χρησιμοποιήσεις για km²
municipalities_4326.geometry.area / 1_000_000
```

Η σωστή αρχή είναι:

> **Ελέγχω πρώτα το CRS και τις μονάδες, και μετά επιλέγω τη γεωμετρική μέτρηση.**


## 18. CRS compatibility σε spatial workflows

Δύο layers μπορούν να εμφανίζονται στην ίδια θέση μέσα σε ένα GIS επειδή το λογισμικό κάνει on-the-fly reprojection. Αυτό δεν σημαίνει ότι τα raw coordinates τους είναι ίδια.

Πριν από spatial operations, distances ή areas, ελέγχουμε πάντα αν τα CRS είναι συμβατά με τη συγκεκριμένη ανάλυση.


In [ ]:
print("EPSG:2100 == EPSG:4326 ?")
print(municipalities.crs == municipalities_4326.crs)

print()
print("Original EPSG:", municipalities.crs.to_epsg())
print("Reprojected EPSG:", municipalities_4326.crs.to_epsg())


## Καλή πρακτική για παρόχους γεωχωρικών δεδομένων

Ένα καλά διανεμημένο spatial dataset πρέπει να συνοδεύεται από:

- έγκυρο CRS metadata μέσα στο αρχείο,
- σαφή EPSG code όπου είναι διαθέσιμος,
- τεκμηρίωση datum/projection και μονάδων,
- αναφορά της γεωγραφικής έκτασης για την οποία προορίζεται το CRS,
- αποφυγή αρχείων με «γνωστό προφορικά» CRS αλλά χωρίς machine-readable metadata.

Το CRS είναι μέρος των δεδομένων, όχι απλώς ρύθμιση εμφάνισης.


## 19. Ασκήσεις

### Άσκηση 1

Εμφάνισε το EPSG code, αν το CRS είναι projected και τη μονάδα του πρώτου άξονα του `municipalities`.

### Άσκηση 2

Μετασχημάτισε το `municipalities` σε EPSG:4326 και εμφάνισε το `total_bounds`.

### Άσκηση 3

Σε μία πρόταση, εξήγησε ποια μέθοδο θα χρησιμοποιούσες σε κάθε περίπτωση:

1. Το dataset έχει σωστές EPSG:2100 coordinates αλλά λείπει το CRS metadata.
2. Το dataset είναι σωστά δηλωμένο ως EPSG:2100 και θέλεις longitude/latitude.

### Άσκηση 4

Υπολόγισε `area_km2` σε αντίγραφο του EPSG:2100 GeoDataFrame και εμφάνισε τους 10 μεγαλύτερους Δήμους.

### Άσκηση 5

Ένα layer δηλώνεται ως EPSG:4326 αλλά τα bounds του είναι περίπου `[100000, 3800000, 1000000, 4600000]`. Τι υποψιάζεσαι και ποιον έλεγχο θα έκανες πριν το χρησιμοποιήσεις;


In [ ]:
# Άσκηση 1


In [ ]:
# Άσκηση 2


In [ ]:
# Άσκηση 3


In [ ]:
# Άσκηση 4


In [ ]:
# Άσκηση 5


## 20. Σύνοψη

Στο PY-08 είδαμε το CRS ως μέρος της σημασίας των γεωμετριών και όχι ως απλό label:

```text
geometry + CRS
      ↓
geographic meaning
      ↓
inspect EPSG + units
      ↓
choose projected or geographic CRS
      ↓
transform correctly with to_crs()
      ↓
measure only in appropriate units
```

Οι τρεις πιο σημαντικές ιδέες είναι:

1. **CRS ≠ coordinates** — το CRS μας λέει πώς να ερμηνεύσουμε τους αριθμούς.
2. **`set_crs()` ≠ `to_crs()`** — το πρώτο δηλώνει metadata, το δεύτερο μετασχηματίζει coordinates.
3. **Οι μονάδες έχουν συνέπειες** — για area/distance δεν αρκεί να υπάρχει κάποιο CRS· πρέπει να είναι κατάλληλο για τη μέτρηση.

Αυτό μας προετοιμάζει για το PY-09, όπου spatial queries και spatial joins απαιτούν σωστή κατανόηση των CRS.


## 21. Λύσεις ασκήσεων

Προσπάθησε πρώτα να λύσεις τις ασκήσεις του **Τμήματος 19** χωρίς να κοιτάξεις παρακάτω.


### Άσκηση 1 — Επιθεώρηση CRS


In [ ]:
print("EPSG:", municipalities.crs.to_epsg())
print("Projected:", municipalities.crs.is_projected)
print("Unit:", municipalities.crs.axis_info[0].unit_name)


### Άσκηση 2 — Reprojection σε EPSG:4326


In [ ]:
exercise_4326 = municipalities.to_crs(epsg=4326)

print(exercise_4326.crs)
print(exercise_4326.total_bounds)


### Άσκηση 3 — `set_crs()` ή `to_crs()`;

1. **`set_crs(epsg=2100)`**: οι coordinates είναι ήδη EPSG:2100 και απλώς λείπει το metadata.
2. **`to_crs(epsg=4326)`**: το source CRS είναι γνωστό και θέλουμε πραγματικό coordinate transformation.


### Άσκηση 4 — 10 μεγαλύτεροι Δήμοι


In [ ]:
exercise_area = municipalities.copy()
exercise_area["area_km2"] = exercise_area.geometry.area / 1_000_000

exercise_area.nlargest(10, "area_km2")[
    ["NAME_GR", "area_km2"]
]


### Άσκηση 5 — Ύποπτα bounds

Για EPSG:4326 περιμένουμε longitude/latitude σε μοίρες, όχι τιμές εκατοντάδων χιλιάδων ή εκατομμυρίων. Αυτό δείχνει ότι πιθανότατα έχει **δηλωθεί λάθος CRS metadata** χωρίς να μετασχηματιστούν οι coordinates.

Πριν χρησιμοποιήσουμε το layer, ελέγχουμε την πηγή/metadata για το πραγματικό CRS και συγκρίνουμε τα bounds με τις αναμενόμενες μονάδες και τη γεωγραφική περιοχή.
